In [4]:
import os
import pandas as pd
import numpy as np
from typing import List, Dict
import import_ipynb
from tokenizer import Tokenizer
from CountVectorFeaturizer import CountVectorFeaturizer
from TfidfFeaturizer import TfidfFeaturizer
from NaiveBayesClassifier import NaiveBayesClassifier

In [5]:
def map_value(value: str) -> str:
    return 'plagiat' if value in ['heavy', 'cut'] else 'non_plagiat'

def load_labels(folder_path: str) -> Dict[str, str]:
    expected_results = pd.read_csv(os.path.join(folder_path, "file_information.csv"), sep=",")
    labels = dict(zip(expected_results["File"], expected_results['Category']))
    return {k: map_value(v) for k, v in sorted(labels.items())}


In [7]:
if __name__ == "__main__":
    folder_path = "../data-plagiarism"

    # Load labels
    filtered_labels = load_labels(folder_path)
    documents = [Tokenizer.read_file(os.path.join(folder_path, file)) for file in filtered_labels.keys()]#read all documents

    # Test both featurizers
    for featurizer_class in [CountVectorFeaturizer, TfidfFeaturizer]:
        featurizer = featurizer_class()
        featurizer.fit(documents)
        matrix = featurizer.transform(documents)

        #word_probabilities = compute_word_probabilities(filtered_labels, featurizer, matrix)
        classifier = NaiveBayesClassifier()
        classifier.fit(filtered_labels, featurizer, matrix)
        
        # Classify files
        results = []
        for file_name in filtered_labels.keys():
            content = Tokenizer.read_file(os.path.join(folder_path, file_name))
            tokens = featurizer.tokenizer.tokenize(content)
            prediction = classifier.predict(tokens)
            results.append({
                "File Name": file_name,
                "Expected Label": filtered_labels[file_name],
                "Predicted Label": prediction
            })

        df_results = pd.DataFrame(results)
        df_results["Mismatch"] = df_results["Expected Label"] != df_results["Predicted Label"]

        print(f"Results for {featurizer_class.__name__}:")
        print(df_results.to_string(index=False))

        mismatches = df_results[df_results["Mismatch"]]
        print(f"Mismatches for {featurizer_class.__name__}:", len(mismatches))
        print(mismatches.to_string(index=False))
        


Results for CountVectorFeaturizer:
     File Name Expected Label Predicted Label  Mismatch
g0pA_taska.txt    non_plagiat     non_plagiat     False
g0pA_taskb.txt        plagiat         plagiat     False
g0pA_taskc.txt    non_plagiat         plagiat      True
g0pA_taskd.txt        plagiat         plagiat     False
g0pA_taske.txt    non_plagiat     non_plagiat     False
g0pB_taska.txt    non_plagiat     non_plagiat     False
g0pB_taskb.txt    non_plagiat     non_plagiat     False
g0pB_taskc.txt        plagiat         plagiat     False
g0pB_taskd.txt    non_plagiat         plagiat      True
g0pB_taske.txt        plagiat         plagiat     False
g0pC_taska.txt        plagiat         plagiat     False
g0pC_taskb.txt    non_plagiat     non_plagiat     False
g0pC_taskc.txt    non_plagiat     non_plagiat     False
g0pC_taskd.txt        plagiat         plagiat     False
g0pC_taske.txt    non_plagiat         plagiat      True
g0pD_taska.txt        plagiat         plagiat     False
g0pD_taskb.tx